In [ ]:
class Cliff:
    def __init__(self):
        self.state = [[0 for _ in range(10)] for _ in range(9)]
        self.state.append([-1] + [2 for _ in range(8)] + [1])
        self.actions = ["u", "d", "l", "r"]
        self.gamma = 0.9

        self.x = 0
        self.y = 9

    def __str__(self):
        display = ""
        for y, row in enumerate(self.state):
            for x, cell in enumerate(row):
                if self.x == x and self.y == y:
                    cell = "P"
                elif cell == 0:
                    cell = "."
                elif cell == -1:
                    cell = "S"
                elif cell == 1:
                    cell = "G"
                elif cell == 2:
                    cell = "X"
                display += f"{cell} "
            display += "\n"

        return display

    def reset(self):
        self.x = 0
        self.y = 9

    def get_state(self):
        return (self.x, self.y)

    def set_state(self, x, y):
        self.x, self.y = x, y

    def get_possive_states(self):
        possive_states = []
        if self.y > 0:
            possive_states.append((self.x, self.y - 1))  # up
        if self.y < 9:
            possive_states.append((self.x, self.y + 1))  # down
        if self.x > 0:
            possive_states.append((self.x - 1, self.y))  # left
        if self.x < 9:
            possive_states.append((self.x + 1, self.y))  # right
        return possive_states
    
    def step(self, act):
        if act == "":
            return (self.x, self.y)
        
        elif act == "u":
            if self.y > 0:
                return (self.x, self.y - 1)
            return (self.x, self.y)
        
        elif act == "d":
            if self.y < 9:
                return (self.x, self.y + 1)
            return (self.x, self.y)
        
        elif act == "l":
            if self.x > 0:
                return (self.x - 1, self.y)
            return (self.x, self.y)
        
        elif act == "r":
            if self.x < 9:
                return (self.x + 1, self.y)
            return (self.x, self.y)

    def possibility(self, act, target_state):
        next_state = self.step(act)
        if next_state == target_state:
            return 1
        return 0

    def action(self, act):
        next_state = self.step(act)
        self.x, self.y = next_state

    def reward(self, action):
        x, y = self.step(action)
        cell = self.state[y][x]
        if cell == 0:
            return -1
        elif cell == -1:
            return -1
        elif cell == 1:
            return 10
        elif cell == 2:
            return -1000

    def is_done(self):
        cell = self.state[self.y][self.x]
        if cell == 1:
            return True
        elif cell == 2:
            return True
        return False
    
    def trace(self, actions):
        self.reset()
        trace = [self.get_state()]
        for action in actions:
            self.action(action)
            trace.append(self.get_state())
        
        board = [["." for _ in range(10)] for _ in range(10)]
        for y in range(10):
            for x in range(10):
                if self.state[y][x] == -1:
                    board[y][x] = "S"
                elif self.state[y][x] == 1:
                    board[y][x] = "G"
                elif self.state[y][x] == 2:
                    board[y][x] = "X"

        for i, (x, y) in enumerate(trace):
            board[y][x] = str(i)

        for row in board:
            print(" ".join(row))

        return 

In [9]:
cliff = Cliff()
print(cliff)

. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
P X X X X X X X X G 



In [10]:
DELTA = 1e-3

In [11]:
state_value = [[0 for _ in range(10)] for _ in range(10)]
policy = [[{"u": 0.25, "d": 0.25, "l": 0.25, "r": 0.25} for _ in range(10)] for _ in range(10)]

state_value_history = []

def policy_evaluation(state_value):
    while True:
        new_state_value = [[0 for _ in range(10)] for _ in range(10)]
        for y in range(10):
            for x in range(10):
                sum_value = 0

                cliff.set_state(x, y)

                if cliff.is_done():
                    new_state_value[y][x] = 0
                    continue

                for action in cliff.actions:
                    policy_poss = policy[y][x][action]
                    reward = cliff.reward(action)

                    transition_value = 0
                    for possive_state in cliff.get_possive_states():
                        transition_poss = cliff.possibility(action, possive_state)
                        transition_value += transition_poss * state_value[possive_state[1]][possive_state[0]]

                    sum_value += policy_poss * (reward + cliff.gamma * transition_value)

                new_state_value[y][x] = sum_value

        max_diff = max(
            abs(state_value[y][x] - new_state_value[y][x]) for y in range(10) for x in range(10)
        )
        state_value = new_state_value
        state_value_history.append(state_value)

        if max_diff < DELTA:
            return state_value

def policy_improvement(policy, state_value):
    policy_stable = True
    for y in range(10):
        for x in range(10):
            old_best_action = max(policy[y][x], key=policy[y][x].get)
            
            action_values = {}
            max_value = -1e9
            best_action = None
            
            for action in cliff.actions:
                cliff.set_state(x, y)
                next_x, next_y = cliff.step(action)
                q_value = cliff.reward(action) + cliff.gamma * state_value[next_y][next_x]
                action_values[action] = q_value

                if q_value > max_value:
                    max_value = q_value
                    best_action = action
            
            if old_best_action != best_action:
                policy_stable = False
            
            for action in cliff.actions:
                if action == best_action:
                    policy[y][x][action] = 1
                else:
                    policy[y][x][action] = 0
                    
    return policy_stable, policy

In [12]:
epoch = 1

while True:
    state_value = policy_evaluation(state_value)
    policy_stable, policy = policy_improvement(policy, state_value)
    print(f"Epoch {epoch} completed.")

    if policy_stable:
        break
    epoch += 1

KeyboardInterrupt: 

In [22]:
for row in state_value_history[-1]:
    print(["{0:0.2f}".format(v) for v in row])

['-1.00', '-1.90', '-2.71', '-3.44', '-4.10', '-4.35', '-3.72', '-3.03', '-2.25', '-1.39']
['-1.90', '-2.71', '-3.44', '-4.10', '-4.35', '-3.72', '-3.03', '-2.25', '-1.39', '-0.43']
['-2.71', '-3.44', '-4.10', '-4.35', '-3.72', '-3.03', '-2.25', '-1.39', '-0.43', '0.63']
['-3.44', '-4.10', '-4.35', '-3.72', '-3.03', '-2.25', '-1.39', '-0.43', '0.63', '1.81']
['-4.10', '-4.35', '-3.72', '-3.03', '-2.25', '-1.39', '-0.43', '0.63', '1.81', '3.12']
['-4.35', '-3.72', '-3.03', '-2.25', '-1.39', '-0.43', '0.63', '1.81', '3.12', '4.58']
['-3.72', '-3.03', '-2.25', '-1.39', '-0.43', '0.63', '1.81', '3.12', '4.58', '6.20']
['-3.03', '-2.25', '-1.39', '-0.43', '0.63', '1.81', '3.12', '4.58', '6.20', '8.00']
['-2.25', '-1.39', '-0.43', '0.63', '1.81', '3.12', '4.58', '6.20', '8.00', '10.00']
['-3.03', '0.00', '0.00', '0.00', '0.00', '0.00', '0.00', '0.00', '0.00', '0.00']


In [23]:
action_trace = []

cliff.reset()
while not cliff.is_done():
    x, y = cliff.get_state()
    action_probs = policy[y][x]
    best_action = max(action_probs, key=action_probs.get)
    cliff.action(best_action)
    action_trace.append(best_action)
    print(best_action)

u
r
r
r
r
r
r
r
r
r
d


In [24]:
cliff.trace(action_trace)

. . . . . . . . . .
. . . . . . . . . .
. . . . . . . . . .
. . . . . . . . . .
. . . . . . . . . .
. . . . . . . . . .
. . . . . . . . . .
. . . . . . . . . .
1 2 3 4 5 6 7 8 9 10
0 X X X X X X X X 11
